In [1]:
import pandas as pd

pd.options.display.float_format = "{:,.2f}".format

vendas = pd.read_csv("Vendas-Globais.csv").dropna(how="all")
vendas["Data"] = pd.to_datetime(vendas["Data"], format="%m/%d/%Y")
vendas["Ano"] = vendas["Data"].dt.year

vendedores = pd.read_csv("vendedores.csv")
fornecedores = pd.read_csv("fornecedores.csv")
transportadoras = pd.read_csv("transportadoras.csv")

vendas = (vendas
          .merge(vendedores, on="VendedorID")
          .merge(fornecedores, on="FornecedorID")
          .merge(transportadoras, on="TransportadoraID"))

print(f"{len(vendas)} registros de vendas, de "
      f"{vendas['Data'].min():%d/%m/%Y} a {vendas['Data'].max():%d/%m/%Y}")

2172 registros de vendas, de 28/02/2009 a 06/11/2012


## 1. Quem são os 10 maiores clientes, em termos de vendas ($)?

In [2]:
vendas.groupby("ClienteNome")["Vendas"].sum().sort_values(ascending=False).head(10)

ClienteNome
Grunewald          201,320.45
Th Fashing         152,114.79
Boleros            131,133.15
Champes             91,362.13
The Corner Store    82,519.82
Eintrach GS         75,153.52
Don Balón           68,265.66
Warp AG             65,815.98
Rode & Vite         59,976.93
Boombastic          50,922.47
Name: Vendas, dtype: float64

## 2. Quais os três maiores países, em termos de vendas ($)?

In [3]:
vendas.groupby("ClientePaís")["Vendas"].sum().sort_values(ascending=False).head(3)

ClientePaís
Germany   519,553.73
USA       186,313.22
France    175,565.30
Name: Vendas, dtype: float64

## 3. Quais as categorias de produtos que geram maior faturamento no Brasil?

In [4]:
brasil = vendas[vendas["ClientePaís"] == "Brazil"]
brasil.groupby("CategoriaNome")["Vendas"].sum().sort_values(ascending=False)

CategoriaNome
Womens wear       60,941.73
Sportwear         19,509.43
Babywear          10,367.21
Men´s Footwear     9,467.74
Ladies´Footwear    7,798.52
Men´s Clothes      6,158.06
Bath Clothes       3,407.70
Children´s wear    2,752.43
Name: Vendas, dtype: float64

## 4. Qual a despesa com frete envolvendo cada transportadora?

In [5]:
vendas.groupby("TransportadoraNome")["Frete"].sum().sort_values(ascending=False)

TransportadoraNome
Global Express     70,897.82
General Shipping   19,843.83
Great Logistics    18,771.69
Name: Frete, dtype: float64

## 5. Principais clientes (vendas $) de Men´s Footwear na Alemanha

In [6]:
calcados_de = vendas[(vendas["CategoriaNome"] == "Men´s Footwear") &
                     (vendas["ClientePaís"] == "Germany")]
calcados_de.groupby("ClienteNome")["Vendas"].sum().sort_values(ascending=False)

ClienteNome
Grunewald            20,488.49
Gluderstedt          11,923.35
Boombastic           11,235.97
Eintrach GS          10,377.83
Warp AG               9,085.76
Noch Einmal GMBH      3,030.72
Casual Clothing       2,602.50
Halle Köln            1,692.56
Man Kleider           1,108.00
Kohl Industries AG      286.56
Name: Vendas, dtype: float64

## 6. Quais os vendedores que mais dão descontos nos Estados Unidos?

In [7]:
eua = vendas[vendas["ClientePaís"] == "USA"]
eua.groupby("VendedorNome")["Desconto"].sum().sort_values(ascending=False)

VendedorNome
Gael Monfils     7,415.13
Yannick Sinner   1,160.05
Martina Hingis     733.78
Cori Gauff         404.55
Name: Desconto, dtype: float64

## 7. Fornecedores com maior margem de lucro ($) em Womens wear

In [8]:
feminino = vendas[vendas["CategoriaNome"] == "Womens wear"]
feminino.groupby("FornecedorNome")["Margem Bruta"].sum().sort_values(ascending=False).head(10)

FornecedorNome
Pälsii Sports    81,839.94
Baby Dress       20,675.22
Wills Surfwear   10,019.12
Great Outdoors    9,165.03
Global Outlet     4,884.99
L.A. Sports       2,304.59
Luis Vilton       2,192.38
Tennis Place      1,845.27
Netshoes          1,810.77
USA Jeans         1,274.00
Name: Margem Bruta, dtype: float64

## 8. Quanto foi vendido em 2009? O faturamento cresce, está estável ou decai?

In [9]:
anual = vendas.groupby("Ano")["Vendas"].sum()
print(anual)
print(f"\nVendido em 2009: $ {anual[2009]:,.2f}")
print("\nVariação ano a ano (%):")
print(anual.pct_change().mul(100).round(1))

Ano
2009    87,666.29
2010   370,788.56
2011   641,719.41
2012   682,973.44
Name: Vendas, dtype: float64

Vendido em 2009: $ 87,666.29

Variação ano a ano (%):
Ano
2009      NaN
2010   323.00
2011    73.10
2012     6.40
Name: Vendas, dtype: float64


**Conclusão:** o faturamento vem **crescendo** todos os anos (2009→2012),
porém o ritmo de crescimento desacelera: +323% em 2010, +73% em 2011 e +6% em 2012.

## 9. Principais clientes de Men´s Footwear em 2013 — cidades e valores

In [10]:
calcados_2013 = vendas[(vendas["CategoriaNome"] == "Men´s Footwear") &
                       (vendas["Ano"] == 2013)]
if calcados_2013.empty:
    print("Não há vendas registradas em 2013 (a base cobre 2009 a 2012).")
else:
    display(calcados_2013.groupby(["ClienteNome", "ClienteCidade"])["Vendas"]
            .sum().sort_values(ascending=False))

Não há vendas registradas em 2013 (a base cobre 2009 a 2012).


## 10. Na Europa, quanto se vende ($) para cada país?

In [11]:
europa = ["France", "Ireland", "UK", "Germany", "Sweden", "Belgium",
          "Spain", "Norway", "Portugal", "Austria", "Switzerland",
          "Finland", "Italy", "Poland", "Denmark"]
vendas[vendas["ClientePaís"].isin(europa)] \
    .groupby("ClientePaís")["Vendas"].sum().sort_values(ascending=False)

ClientePaís
Germany       519,553.73
France        175,565.30
UK            167,101.35
Ireland       131,133.15
Denmark        59,976.93
Sweden         57,162.01
Austria        54,468.49
Spain          28,460.23
Portugal       14,230.55
Belgium        13,952.71
Switzerland    12,855.03
Italy          12,205.31
Finland         6,457.66
Poland          3,977.18
Norway            193.36
Name: Vendas, dtype: float64